In [18]:
# ========== 导入：第 1 周练习 —— 本地 Llama 技术问答（流式 / 非流式）==========
# 练习目标：用 Ollama 的 llama3.2 回答技术问题；对比 OpenAI 兼容流式 vs ollama 库非流式
# 怎么跑：确保 Ollama 已启动并 pull 了 llama3.2，然后自上而下运行各格

# 从 IPython.display 导入展示工具：Markdown 渲染、display 新建输出、update_display 原地刷新（流式打字机）
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端：走 Ollama 的 /v1 兼容 Chat Completions
from openai import OpenAI
# 导入 ollama 官方 Python 库：后面用 ollama.chat 做非流式对照
import ollama


In [19]:
# ========== 常量：模型名集中写一处，后面只改这里 ==========

# 本地 Ollama 模型名；字符串必须与本机 `ollama list` 里的名字一致
MODEL_LLAMA = 'llama3.2'


In [20]:
# ========== 配置 Ollama 客户端（OpenAI 兼容，无需真实云端 API Key）==========

# base_url 指向本机 Ollama 的 OpenAI 兼容层；api_key 仅占位，本地会忽略
ollama_client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)


In [21]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的问题保留英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天卡住的技术问题，再跑下面流式 / 非流式两格对比
question = """
How many parameters do you have?
"""


In [22]:
# ========== 提示词：system 定角色，user 拼上具体问题 ==========

# system_prompt：告诉模型「你是谁、答什么领域」；英文指令保留不译
system_prompt = "You are a helpful technical assistant who answers questions about llm and agenticAI"
# user_prompt：固定英文前缀 + 上面的 question
user_prompt = "Please give a correct answer to the following question: " + question


In [23]:
# ========== messages：Chat Completions 标准的 system + user 列表 ==========

# 两条消息组成对话上下文，后面流式 / 非流式共用这份 messages
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]


In [24]:
# ========== 路径 A：用 Llama 3.2 流式回答（经 OpenAI 兼容客户端）==========

# stream=True：服务端边生成边推增量；客户端可边收边刷新显示
stream = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    stream=True
)

# response：累积到目前为止的完整文本
response = ""
# 先放一个空 Markdown，拿到 display_id，便于后面原地 update（打字机效果）
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    # delta.content 可能是 None（某些 chunk 只有角色/结束标记），用 or '' 兜底
    response += chunk.choices[0].delta.content or ''
    # 去掉可能出现的 markdown 代码围栏噪声，避免渲染成代码块
    response = response.replace("```", "").replace("markdown", "")
    # 用同一 display_id 刷新，实现边生成边更新
    update_display(Markdown(response), display_id=display_handle.display_id)


I'm an LLaMA model, which stands for Long Short-Term Architecture MAssive transformer. I have approximately 337 billion parameters, making me one of the largest language models available today!

In [25]:
# ========== 路径 B：用 Llama 3.2 非流式回答（直接调用 ollama 库）==========

# ollama.chat：等整段生成完再返回 dict；与上面流式路径对照体验差异
response = ollama.chat(model=MODEL_LLAMA, messages=messages)
# 原生 API 的回复正文在 response['message']['content']
reply = response['message']['content']
# 一次性用 Markdown 展示完整回答
display(Markdown(reply))


I'm an LLaMA model, which stands for Large Language Model. According to my architecture, I have approximately 452 billion parameters.

Note that this number is constantly changing as new updates are applied and fine-tuned, so it's possible that the exact count may vary in the future. However, 452 billion is a commonly cited estimate based on publicly available information about LLaMA models like myself.

In [26]:
# ========== 交互模式：运行后在输入框里键入你自己的问题 ==========

# input()：阻塞等待你在终端/笔记本里输入；提示语保留英文（可运行字符串）
my_question = input("Please enter your question: ")

# 仍用同一个 system_prompt；user 侧换成「详细解释」前缀 + 你刚输入的问题
my_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Please give a detailed explanation to the following question: " + my_question}
]

# 再次走 OpenAI 兼容流式接口
stream = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=my_messages,
    stream=True
)

# result：累积流式增量
result = ""
# 新建可刷新的 Markdown 显示句柄
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    # 拼接增量；None 当空串
    result += chunk.choices[0].delta.content or ''
    # 同样清洗可能的代码围栏标记
    result = result.replace("```", "").replace("markdown", "")
    # 原地刷新显示
    update_display(Markdown(result), display_id=display_handle.display_id)


To provide an accurate answer, we need to understand what "tokenization" means and how it's implemented.

**Tokenization**

Tokenization is the process of breaking down text into individual tokens, also known as subwords or word pieces. These tokens are the basic building blocks of natural language processing (NLP) models like myself.

**What are token types?**

There are several token types used in NLP:

1. **Wordpiece**: a contiguous sequence of characters within a word, e.g., "hello" -> ["hel", "lo"]
2. **Subword**: a subsequence of characters within a word, e.g., "un" from the word "under"
3. **Character token**: an individual character in the text

**How many tokens are created for a sentence?**

When I tokenize a sentence, I break it down into individual words or subwords based on a set of predefined rules and algorithms. The number of tokens created can vary depending on the tokenizer used and the specific requirements of the NLP model.

Here's a simplified overview of how tokenization works:

1. **Splitting**: Divide the input text into subword units (e.g., wordpieces or subwords) based on a set of rules, such as:
	* BPE (Byte Pair Encoding): splits characters at the beginning and end of each subword.
	* WordPiece: splits words into smaller subwords.
2. **Insertion symbols**: Special tokens like `[UNK]`, `</s>`, and `</p>` are inserted to mark the beginning or end of sequences, as well as to handle out-of-vocabulary (OOV) words.

Now, let's apply this to your original question: "how many token do you tokenize in this sentence?"

Here's a breakdown for a relatively common tokenization scheme:

1. Single space character (` `): 1 token
2. Punctuation marks:
	* `.`: 1 token (.) 
	* `,`: 1 token (,)
	* `'`: 1 token (')
3. Most words: Wordpiece or subword, e.g., "Tokenization" breaks down into ["Toe", "knal",..."iazation"]
4. Special tokens:
	* `[/s]`: ends the sequence
	* `[/p]`: separates sequence blocks (if applicable)

Let's assume an average sentence with 15 words and 5-10 subwords per word. The total number of tokens would be:

1. Words: approximately 15 × ∞ tokens (wordpiece/subword) ≈ 45-90 tokens
2. Special tokens: 3-4 tokens (∼ `/s/`, `[/p]/`)
3. Punctuation: add about 5-10 tokens
4. Space: 1 token

This simplifies to approximately **50-60** unique tokens for the original sentence.

However, this is just an estimate and can vary greatly depending on the specific tokenizer used, the language model's requirements, and the desired level of subword granularity.

If you want a more accurate count or have further questions about your particular case, feel free to share more details!